<a href="https://colab.research.google.com/github/jiyeonlee-2930/DeepLearning-TensorFlow-Basic/blob/main/Knowledge_Distrillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
from google.colab.patches import cv2_imshow
from tqdm import tqdm


In [ ]:
#@title 파라미터 설정
t_ephoc = 31 #@param {type:"slider", min:1, max:100, step:1}
s_ephoc = 13 #@param {type:"slider", min:1, max:100, step:1}
learning_rate = 0.01
batch_size = 64 #@param [32, 64, 128, 256] {type:"raw"}
temperature = 4 #@param {type:"slider", min:1, max:10, step:1}
alpha = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}

In [ ]:
# MNIST 데이터셋 가져오기

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_train = np.reshape(x_train, (-1, 28, 28, 1))

x_test = x_test.astype("float32") / 255.0
x_test = np.reshape(x_test, (-1, 28, 28, 1))

In [ ]:
# teacher 모델
i=tf.keras.Input(shape=(28, 28, 1))
out=tf.keras.layers.Conv2D(256, (3, 3), strides=(2, 2), padding="same")(i)
out=tf.keras.layers.LeakyReLU(alpha=0.2)(out)
out=tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(1, 1), padding="same")(out)
out=tf.keras.layers.Conv2D(512, (3, 3), strides=(2, 2), padding="same")(out)
out=tf.keras.layers.Flatten()(out)
out=tf.keras.layers.Dense(10)(out)
t_model=tf.keras.Model(inputs=[i],outputs=[out])

t_model.summary()

In [ ]:
# student 모델
i=tf.keras.Input(shape=(28, 28, 1))
out=tf.keras.layers.Flatten()(i)
out=tf.keras.layers.Dense(28)(out)
out=tf.keras.layers.Dense(10)(out)

s_model_1=tf.keras.Model(inputs=[i],outputs=[out])
s_model_2=tf.keras.models.clone_model(s_model_1)

s_model_1.summary()

In [ ]:
# teacher 모델
t_model.compile(tf.keras.optimizers.Adam(learning_rate),
                tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

# student 모델 (distillation 적용)
s_model_1.compile(tf.keras.optimizers.Adam(learning_rate),
                tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

# 비교 모델 (distillation 미적용)
s_model_2.compile(tf.keras.optimizers.Adam(learning_rate),
                tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

In [ ]:
# teacher 모델
t_model.fit(x_train, y_train,batch_size=batch_size,epochs=t_ephoc)

In [ ]:
# student 손실함수
s_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# distillation 손실함수
d_loss = tf.keras.losses.KLDivergence()

In [ ]:
x_train.shape

In [ ]:
batch_count =x_train.shape[0]//batch_size

opt_s1 = tf.keras.optimizers.Adam(learning_rate) # Optimizer for s_model_1
opt_s2 = tf.keras.optimizers.Adam(learning_rate) # Optimizer for s_model_2

for e in range(s_ephoc):
    for _ in range(batch_count):
        batch_num=np.random.randint(0, x_train.shape[0], size=batch_size)
        t_pred = t_model.predict(x_train[batch_num])

        with tf.GradientTape() as tape:
            s_pred_1 = s_model_1(x_train[batch_num])
            student_loss = s_loss(y_train[batch_num], s_pred_1)
            distillation_loss = d_loss(
                tf.nn.softmax(t_pred / temperature, axis=1),
                tf.nn.softmax(s_pred_1 / temperature, axis=1),
            )
            loss = alpha * student_loss + (1 - alpha) * distillation_loss

        vars_s1 = s_model_1.trainable_variables
        grad_s1 = tape.gradient(loss, vars_s1)
        opt_s1.apply_gradients(zip(grad_s1, vars_s1))

        with tf.GradientTape() as tape:
            s_pred_2 = s_model_2(x_train[batch_num])
            student_loss = s_loss(y_train[batch_num], s_pred_2)
        vars_s2 = s_model_2.trainable_variables
        grad_s2 = tape.gradient(student_loss, vars_s2)
        opt_s2.apply_gradients(zip(grad_s2, vars_s2))

    print("에포크 {}".format(e))
    print("선생님께 배운 경우")
    s_model_1.evaluate([x_test], y_test)
    print("혼자 공부한 경우")
    s_model_2.evaluate([x_test], y_test)
    print("\n")